## Read the file

In [1]:
file_path = "../data/private/fine_tuning.txt"
with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

len(lines)

44777

## Clean the conversation

In [2]:
import re

encryption_message = "Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more."
media_pattern = "<Multimedia omitido>"
email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}'
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
edited_message = "<Se editó este mensaje.>"
deleted_message = "Eliminaste este mensaje."
null_message = "null"
created_group_message = "created group"
added_you_to_group_message = "added you"
tagging_pattern = r'@[\w]+'


filtered_lines = []
for line in lines:
    if (
            encryption_message not in line and
            deleted_message not in line and
            null_message != line.split(" ")[-1] and
            media_pattern not in line and
            created_group_message not in line and
            added_you_to_group_message not in line and
            not re.search(email_pattern, line) and
            not re.search(url_pattern, line)
    ):
        line = line.replace(edited_message, "").strip()
        line = re.sub(tagging_pattern, "", line).strip()
        filtered_lines.append(line)

pattern = r'(\d{1,2}/\d{1,2}/\d{2,4}, \d{1,2}:\d{2}(?::\d{2})?(?:\s?[APap][Mm])?)\s?(?:-|\~)?\s?(.*?): (.*?)(?=\n\d{1,2}/\d{1,2}/\d{2,4}, \d{1,2}:\d{2}|$)'
content = '\n'.join(filtered_lines)
messages = re.findall(pattern, content, re.DOTALL)

lines_removed = len(lines) - len(filtered_lines)
print(f"Lines removed: {lines_removed}")
#print(f"Messages: {messages}")

Lines removed: 19557


## Create the dataset

### 1. Group messages by sender

If a conversation is structured as follows:  

```
User 1: Hey!  
User 1: How are you?  
User 2: I am fine  
User 2: And you?  
User 1: Good.  
```

We want to transform it into:  

```
User 1: Hey!\nHow are you? 
User 2: I am fine\nAnd you?  
User 1: Good  
```

In [3]:
grouped_messages = []
same_day_messages = []
last_date = None

for _, sender, message in messages:
    date=_.split(',')[0]
    if last_date is None:
        last_date = date

    if date == last_date:
        if same_day_messages and same_day_messages[-1]["sender"] == sender:
            #same_day_messages.append({
            #    "sender": sender,
            #    "message": message
            #})
            same_day_messages[-1]["message"] += "\n" + message
        else:
            same_day_messages.append({
                "sender": sender,
                "message": message
            })
    else:
        grouped_messages.append(same_day_messages)
        same_day_messages = [{
            "sender": sender,
            "message": message
        }]
        last_date = date
if same_day_messages:
    grouped_messages.append(same_day_messages)

len(grouped_messages)
print(grouped_messages)

[[{'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'We?\nTe muriste?'}], [{'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'Aye\nOye*\nSobre black cover\nComo que la madre de asta crio al demonio que está dentro del grimorio de asta\nHas leído eso no?'}, {'sender': 'Pedro Guill Ferri', 'message': 'Si'}, {'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'Vale'}, {'sender': 'Pedro Guill Ferri', 'message': 'Por cierto andres preparate para tu masacre por hacerme spoiler'}, {'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'Q?\nPero si te pregunto si has leído esa parte del manga\nY me dices que si\nPero ahora me dices q me matas?'}, {'sender': 'Pedro Guill Ferri', 'message': 'Yo me refería a tu mensaje\nPto'}, {'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'A\nPues eso\nY el demonio los quiere violar a todos los demonios'}, {'sender': 'Pedro Guill Ferri', 'message': '.'}, {'sender': 'La Wikipedia Del Porno(Andres)', 'message': 'Por que el señor lucifer i

### 2. Include special tokens

Each message follows this format:  
```
<|startoftext|>Sender<|separator|>Message<|endoftext|>
```

In [4]:
# Define special tokens
start_of_text_token = "<|startoftext|>"
end_of_text_token = "<|endoftext|>"
separator_token = "<|separator|>"

fine_tuning_data = []

for day in grouped_messages:
    day_sequences = []
    for message in day:
        sender = message["sender"]
        role=""
        if sender=="Pedro Guill Ferri":
            role="user"
        else:
            role="assistant"
        message_text = message["message"]
        input_sequence = f"{start_of_text_token}{sender}{separator_token}{message_text}{end_of_text_token}"
        day_sequences.append({"role": role,"content":message_text})
    fine_tuning_data.append(day_sequences)

len(fine_tuning_data)

1112

In [4]:
# Define special tokens
start_of_text_token = "<|startoftext|>"
end_of_text_token = "<|endoftext|>"
separator_token = "<|separator|>"

fine_tuning_data = []

for day in grouped_messages:
    day_sequences = []
    for message in day:
        sender = message["sender"]
        message_text = message["message"]
        input_sequence = f"{start_of_text_token}{sender}{separator_token}{message_text}{end_of_text_token}"
        day_sequences.append(input_sequence)
    fine_tuning_data.append(day_sequences)

len(fine_tuning_data)

1112

### 3. Save the data

In [5]:
import json

save_path = "../output/fine_tuning/data/fine_tuning_pedro.json"
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(fine_tuning_data, f, ensure_ascii=False, indent=4)